# Advanced Stacking Pipeline for Top-10 Ranking
Multi-model ensemble with feature engineering & meta-learner

## Cell 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error
import gc
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("✓ All libraries loaded")

## Cell 2: Load & Prepare Data

In [ ]:
# Load data
print("Loading data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

# Compress dtypes
def compress_dtypes(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_df = compress_dtypes(train_df)
test_df = compress_dtypes(test_df)

# Clean data
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

y_train = train_df['TARGET'].values.copy()
test_ids = test_df['ID'].values.copy()

X_train = train_df.drop(['ID', 'TARGET'], axis=1)
X_test = test_df.drop(['ID'], axis=1)

# Remove constant columns
const_cols = list(set(X_train.columns[X_train.nunique() <= 1]) | set(X_test.columns[X_test.nunique() <= 1]))
X_train = X_train.drop(const_cols, axis=1, errors='ignore')
X_test = X_test.drop(const_cols, axis=1, errors='ignore')

# Align columns
common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

del train_df, test_df
gc.collect()

print(f"✓ Train: {X_train.shape}, Test: {X_test.shape}")
print(f"✓ Target mean: {y_train.mean():.4f}, std: {y_train.std():.4f}")

## Cell 3: Feature Engineering

In [ ]:
# Create statistical features
def engineer_features(X):
    X_feat = X.copy()
    
    # Row-wise statistics
    X_feat['row_mean'] = X.mean(axis=1)
    X_feat['row_std'] = X.std(axis=1)
    X_feat['row_median'] = X.median(axis=1)
    X_feat['row_skew'] = X.skew(axis=1)
    X_feat['row_max'] = X.max(axis=1)
    X_feat['row_min'] = X.min(axis=1)
    X_feat['row_max_min_diff'] = X_feat['row_max'] - X_feat['row_min']
    
    # Lag interaction features (for LagT* columns)
    lag_cols = [c for c in X.columns if 'LagT' in c]
    if len(lag_cols) > 0:
        lag_subset = X[lag_cols]
        X_feat['lag_mean'] = lag_subset.mean(axis=1)
        X_feat['lag_std'] = lag_subset.std(axis=1)
    
    return X_feat

print("Engineering features...")
X_train_feat = engineer_features(X_train)
X_test_feat = engineer_features(X_test)

print(f"✓ Features expanded: {X_train.shape[1]} -> {X_train_feat.shape[1]}")

## Cell 4: Setup Stacking Framework

In [ ]:
# 5-fold CV for stacking
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_indices = list(kf.split(X_train_feat))

# Initialize OOF and test predictions for each base model
n_models = 6  # LGB, XGB, CB, Ridge, ElasticNet, Lasso
oof_preds = np.zeros((len(X_train_feat), n_models))
test_preds = np.zeros((len(X_test_feat), n_models))
cv_scores = []

print(f"✓ 5-Fold CV stacking setup")
print(f"✓ {n_models} base models")
print(f"✓ Meta-features for stacking: ({len(X_train_feat)}, {n_models})")

## Cell 5: Model 1 - LightGBM

In [ ]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_data_in_leaf': 20,
    'lambda_l1': 0.5,
    'lambda_l2': 0.5,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'random_state': 42,
    'verbose': -1,
    'num_threads': 4
}

lgb_cv_scores = []
for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    X_tr, X_val = X_train_feat.iloc[train_idx], X_train_feat.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val)
    
    model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=800,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(period=0)]
    )
    
    oof_preds[val_idx, 0] = model.predict(X_val)
    test_preds[:, 0] += model.predict(X_test_feat) / 5
    
    r2 = r2_score(y_val, oof_preds[val_idx, 0])
    lgb_cv_scores.append(r2)
    print(f"Fold {fold_num} LightGBM R²: {r2:.6f}")
    del model, X_tr, X_val, y_tr, y_val
    gc.collect()

print(f"✓ LightGBM Avg R²: {np.mean(lgb_cv_scores):.6f}")

## Cell 6: Model 2 - XGBoost

In [ ]:
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'lambda': 1.0,
    'alpha': 0.5,
    'random_state': 42,
    'n_jobs': 4,
    'verbosity': 0
}

xgb_cv_scores = []
for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    X_tr, X_val = X_train_feat.iloc[train_idx], X_train_feat.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    model = xgb.XGBRegressor(**xgb_params, n_estimators=800)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=50,
        verbose=False
    )
    
    oof_preds[val_idx, 1] = model.predict(X_val)
    test_preds[:, 1] += model.predict(X_test_feat) / 5
    
    r2 = r2_score(y_val, oof_preds[val_idx, 1])
    xgb_cv_scores.append(r2)
    print(f"Fold {fold_num} XGBoost R²: {r2:.6f}")
    del model, X_tr, X_val, y_tr, y_val
    gc.collect()

print(f"✓ XGBoost Avg R²: {np.mean(xgb_cv_scores):.6f}")

## Cell 7: Model 3 - CatBoost

In [ ]:
cb_params = {
    'iterations': 500,
    'depth': 6,
    'learning_rate': 0.05,
    'l2_leaf_reg': 3,
    'random_state': 42,
    'verbose': False,
    'task_type': 'CPU',
    'thread_count': 4
}

cb_cv_scores = []
for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    X_tr, X_val = X_train_feat.iloc[train_idx], X_train_feat.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    model = cb.CatBoostRegressor(**cb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=30,
        verbose=False
    )
    
    oof_preds[val_idx, 2] = model.predict(X_val)
    test_preds[:, 2] += model.predict(X_test_feat) / 5
    
    r2 = r2_score(y_val, oof_preds[val_idx, 2])
    cb_cv_scores.append(r2)
    print(f"Fold {fold_num} CatBoost R²: {r2:.6f}")
    del model, X_tr, X_val, y_tr, y_val
    gc.collect()

print(f"✓ CatBoost Avg R²: {np.mean(cb_cv_scores):.6f}")

## Cell 8: Model 4-6 - Linear Models (Ridge, ElasticNet, Lasso)

In [ ]:
# Scale features for linear models
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)
X_test_scaled = scaler.transform(X_test_feat)

ridge_scores = []
elastic_scores = []
lasso_scores = []

for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    X_tr = X_train_scaled[train_idx]
    X_val = X_train_scaled[val_idx]
    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]
    
    # Ridge
    model_ridge = Ridge(alpha=0.5)
    model_ridge.fit(X_tr, y_tr)
    oof_preds[val_idx, 3] = model_ridge.predict(X_val)
    test_preds[:, 3] += model_ridge.predict(X_test_scaled) / 5
    r2 = r2_score(y_val, oof_preds[val_idx, 3])
    ridge_scores.append(r2)
    
    # ElasticNet
    model_elastic = ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42, max_iter=10000)
    model_elastic.fit(X_tr, y_tr)
    oof_preds[val_idx, 4] = model_elastic.predict(X_val)
    test_preds[:, 4] += model_elastic.predict(X_test_scaled) / 5
    
    # Lasso
    model_lasso = Lasso(alpha=0.0001, random_state=42, max_iter=10000)
    model_lasso.fit(X_tr, y_tr)
    oof_preds[val_idx, 5] = model_lasso.predict(X_val)
    test_preds[:, 5] += model_lasso.predict(X_test_scaled) / 5
    
    if fold_num == 1:
        print(f"Fold {fold_num} Ridge R²: {r2:.6f}")

print(f"✓ Ridge Avg R²: {np.mean(ridge_scores):.6f}")
print(f"✓ ElasticNet & Lasso trained")

## Cell 9: Base Model Performance

In [ ]:
print("\n" + "="*70)
print("BASE MODELS CROSS-VALIDATION PERFORMANCE")
print("="*70)

model_names = ['LightGBM', 'XGBoost', 'CatBoost', 'Ridge', 'ElasticNet', 'Lasso']
all_scores = [lgb_cv_scores, xgb_cv_scores, cb_cv_scores, ridge_scores, elastic_scores, lasso_scores]

for name, scores in zip(model_names, all_scores):
    print(f"{name:12} | Mean R²: {np.mean(scores):.6f} | Std: {np.std(scores):.6f}")

print("="*70)

## Cell 10: Train Meta-Learner (Stack)

In [ ]:
# Train meta-learner on OOF predictions
print("\nTraining meta-learner on OOF predictions...")

# Use another 5-fold CV for meta-learner
meta_oof = np.zeros(len(X_train_feat))
meta_test = np.zeros(len(X_test_feat))
meta_scores = []

for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    X_meta_tr = oof_preds[train_idx]
    X_meta_val = oof_preds[val_idx]
    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]
    
    # Meta-learner: Ridge on base model OOF predictions
    meta_model = Ridge(alpha=0.1)
    meta_model.fit(X_meta_tr, y_tr)
    
    meta_oof[val_idx] = meta_model.predict(X_meta_val)
    meta_test += meta_model.predict(test_preds) / 5
    
    r2 = r2_score(y_val, meta_oof[val_idx])
    meta_scores.append(r2)
    print(f"Fold {fold_num} Meta-Learner R²: {r2:.6f}")

print(f"\n✓ Meta-Learner (Stacking) R²: {np.mean(meta_scores):.6f}")

final_pred = meta_test

## Cell 11: Generate Submission

In [ ]:
submission = pd.DataFrame({
    'ID': test_ids,
    'TARGET': final_pred
})

print(f"\nSubmission shape: {submission.shape}")
print(f"Target stats:")
print(f"  Mean: {submission['TARGET'].mean():.6f}")
print(f"  Std:  {submission['TARGET'].std():.6f}")
print(f"  Min:  {submission['TARGET'].min():.6f}")
print(f"  Max:  {submission['TARGET'].max():.6f}")

print(f"\nFirst 10 rows:")
print(submission.head(10))

submission.to_csv('submission.csv', index=False)
print(f"\n✓✓✓ Stacking submission saved! ✓✓✓")

## Cell 12: Final Summary

In [ ]:
print("\n" + "="*70)
print("ADVANCED STACKING PIPELINE - SUBMISSION READY")
print("="*70)
print(f"\n🔧 Feature Engineering:")
print(f"  - Row-wise statistics (mean, std, median, skew, max, min)")
print(f"  - Lag feature aggregations")
print(f"  - Total features: {X_train_feat.shape[1]}")
print(f"\n📊 Base Models (5-Fold CV):")

for name, scores in zip(model_names, all_scores):
    print(f"  {name:12}: {np.mean(scores):.6f}")

print(f"\n🎯 Meta-Learner: Ridge on OOF predictions")
print(f"  Stacking R²: {np.mean(meta_scores):.6f}")
print(f"\n📁 Submission:")
print(f"  File: submission.csv")
print(f"  Rows: {len(submission)}")
print(f"\n🚀 Strategy for Top-10:")
print(f"  1. Ensemble diversity: Tree + Linear models")
print(f"  2. Stacking with meta-learner (reduces overfit)")
print(f"  3. Feature engineering for better signal")
print(f"  4. Conservative hyperparameters (avoid CV overfitting)")
print(f"\n" + "="*70)